# RSNA Knee — Colab setup (Spec 01)

Runs Spec 01 end to end on Colab with the data on Google Drive.

**Layout this notebook assumes**

| What | Where | Why |
|---|---|---|
| Repo + artifacts | `/content/drive/MyDrive/rsna-knee` | survives the session; `folds.parquet` must never be regenerated |
| Competition data | `/content/drive/MyDrive/rsna-data` | 570 GB, written once |
| Cache during training (Spec 03+) | `/content/cache` (local disk) | Drive is far too slow to train off |

Sessions are wiped on disconnect, so **every cell here is safe to re-run**. The header
sweep in step 5 checkpoints its progress and resumes where it stopped.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the repo onto Drive

Clone once; afterwards this pulls. Replace the URL with your own remote.

In [ ]:
REPO = '/content/drive/MyDrive/rsna-knee'
REMOTE = 'https://github.com/LIMAMMohamedlimam/rsna-knee.git'  # HTTPS — SSH keys are not available here

import os, subprocess
from pathlib import Path


def git(*args, check=True):
    """Run git, ALWAYS showing stderr and raising on failure — a swallowed pull error leaves
    stale code running and misattributes every later failure."""
    result = subprocess.run(['git', *args], capture_output=True, text=True)
    for stream in (result.stdout, result.stderr):
        if stream.strip():
            print(stream.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed (exit {result.returncode})")
    return result.stdout.strip()


if not Path(REPO).exists():
    if not REMOTE:
        raise SystemExit(f'{REPO} not found — set REMOTE, or copy the repo folder into Drive')
    git('clone', REMOTE, REPO)
elif REMOTE:
    # Unlike the Kaggle copy this one lives on Drive and may hold work you care about, so
    # pull rather than reset — but fail loudly instead of silently staying behind.
    git('-C', REPO, 'pull', '--ff-only', check=False)

os.chdir(REPO)
print('\nHEAD:', git('-C', REPO, 'rev-parse', '--short', 'HEAD'))

## 3. Environment variables

`%env` sets them for the kernel. `!export` would die with its subshell and do nothing.

In [ ]:
%env RSNA_RAW=/content/drive/MyDrive/rsna-data
%env RSNA_ARTIFACTS=/content/drive/MyDrive/rsna-knee/artifacts

## 4. Install dependencies and verify

Re-run after every reconnect — Colab wipes installed packages.

In [ ]:
!python scripts/colab_bootstrap.py --no-mount

In [ ]:
!make test PYTEST=pytest

## 5. EDA (Spec 01 Task 1.2)

Reads one DICOM header per study to recover `PatientID` — this is what lets folds group
by patient instead of by study. Headers are a few KB each, but Drive answers one file at a
time, so expect this to be the slow step.

**If the session dies, just re-run this cell.** Progress is checkpointed to
`artifacts/eda/study_headers.parquet` every 500 studies.

In [ ]:
!make eda PY=python

Check the PatientID coverage before continuing — below 95% and the folds fall back to
study-level grouping, which lets one patient span two folds.

In [ ]:
import pandas as pd
meta = pd.read_parquet('artifacts/eda/study_meta.parquet')
print('studies          :', len(meta))
print('PatientID coverage:', f"{meta['PatientID'].notna().mean():.1%}")
print('site cluster src :', meta['site_cluster_source'].value_counts().to_dict())

## 6. Frozen folds (Spec 01 Task 1.3)

Runs **once for the whole competition**. Re-running exits non-zero on purpose.

In [ ]:
!make folds PY=python

In [ ]:
folds = pd.read_parquet('artifacts/folds.parquet')
print(folds['fold'].value_counts().sort_index())
print('labeled studies:', int(folds['has_gt_labels'].sum()), '/', len(folds))
folds.head()

## 7. Read the report

Two things to look at before Spec 02: the **language table** (Spec 02 needs ≥2 few-shot
examples per major language) and the **report length percentiles** (they set the LLM cost
estimate). Then read the risks section.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('docs/eda_report.md').read()))